# NeuMF Recommender System — MovieLens 100K
**Implicit Feedback · Negative Sampling · NDCG@K · HitRate@K · LLM Cold-Start**

Architecture: Neural Matrix Factorization (He et al., WWW 2017)  
Loss: Binary Cross-Entropy with Logits  
Evaluation: Leave-one-out with 99 sampled negatives per user


In [ ]:
# ── Cell 1: Dependencies ───────────────────────────────────────────────────────
# Uncomment if running on Colab/fresh environment
# !pip install -q torch anthropic


In [ ]:
# ── Cell 2: Imports & Global Seed ─────────────────────────────────────────────
import os, io, json, zipfile, random, warnings
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 1. Data Loading & Preprocessing

In [ ]:
# ── Cell 3: Load MovieLens 100K ───────────────────────────────────────────────
def load_movielens_100k(cache_dir: str = 'ml-100k') -> tuple:
    """
    Download and parse MovieLens 100K.
    Returns:
        ratings  : DataFrame [user_id, item_id, rating, timestamp]
        items_df : DataFrame with item_id, title, and binary genre columns
    """
    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
    if not os.path.exists(cache_dir):
        print("Downloading MovieLens 100K...")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall('.')
        print("Done.")

    ratings = pd.read_csv(
        f'{cache_dir}/u.data', sep='\t',
        names=['user_id', 'item_id', 'rating', 'timestamp']
    )

    genre_cols = [
        'unknown','action','adventure','animation','childrens','comedy',
        'crime','documentary','drama','fantasy','film_noir','horror',
        'musical','mystery','romance','sci_fi','thriller','war','western'
    ]
    items_df = pd.read_csv(
        f'{cache_dir}/u.item', sep='|', encoding='latin-1', header=None,
        usecols=list(range(24)),
        names=['item_id','title','release_date','video_date','imdb_url'] + genre_cols
    )[['item_id', 'title'] + genre_cols]

    return ratings, items_df


ratings, items_df = load_movielens_100k()
print(f"Ratings : {len(ratings):,}")
print(f"Users   : {ratings.user_id.nunique()}")
print(f"Items   : {ratings.item_id.nunique()}")
print(f"Rating distribution:\n{ratings['rating'].value_counts().sort_index()}")

In [ ]:
# ── Cell 4: Binarize & Leave-One-Out Split ────────────────────────────────────
def build_implicit_dataset(
    ratings: pd.DataFrame,
    pos_threshold: int = 4,
    min_interactions: int = 5
) -> tuple:
    """
    Convert explicit ratings → implicit binary feedback, then split.

    Binarization rule:
        rating >= pos_threshold  →  positive interaction (kept)
        rating <  pos_threshold  →  discarded
        (We treat absence-of-preference as no signal, not as negative.)

    Split strategy: Leave-One-Out (LOU)
        - Sort each user's positives by timestamp.
        - Hold out the LAST positive as the test item.
        - All earlier positives form the training set.

    Returns:
        train_df  : DataFrame [user, item, label=1]
        test_df   : DataFrame [user, item]  — one row per user
        user_pos  : dict  user → set of training-positive item indices
        n_users   : int
        n_items   : int
        item2idx  : dict  original_item_id → 0-based index
    """
    # 1. Binarize: keep only positives
    pos = ratings[ratings['rating'] >= pos_threshold].copy()

    # 2. Re-index users and items to 0-based integers
    user_ids  = sorted(pos['user_id'].unique())
    item_ids  = sorted(pos['item_id'].unique())
    user2idx  = {u: i for i, u in enumerate(user_ids)}
    item2idx  = {v: i for i, v in enumerate(item_ids)}
    pos['user'] = pos['user_id'].map(user2idx)
    pos['item'] = pos['item_id'].map(item2idx)

    n_users = len(user2idx)
    n_items = len(item2idx)

    # 3. Remove cold-start users
    pos = pos.groupby('user').filter(lambda g: len(g) >= min_interactions)
    pos = pos.sort_values(['user', 'timestamp'])

    # 4. Leave-one-out split
    train_rows, test_rows = [], []
    user_pos = defaultdict(set)   # user → set of train-positive item indices

    for uid, grp in pos.groupby('user'):
        items_sorted = grp['item'].tolist()
        # Last item → test
        test_item = items_sorted[-1]
        test_rows.append({'user': uid, 'item': test_item})
        # All earlier items → train
        for it in items_sorted[:-1]:
            train_rows.append({'user': uid, 'item': it, 'label': 1})
            user_pos[uid].add(it)

    train_df = pd.DataFrame(train_rows)
    test_df  = pd.DataFrame(test_rows)

    return train_df, test_df, user_pos, n_users, n_items, item2idx


train_df, test_df, user_pos, n_users, n_items, item2idx = build_implicit_dataset(
    ratings, pos_threshold=4, min_interactions=5
)

print(f"Train positives : {len(train_df):,}")
print(f"Test users      : {len(test_df)}")
print(f"n_users={n_users}, n_items={n_items}")
print(f"Avg positives/user (train): {len(train_df)/n_users:.1f}")

## 2. Dataset with Online Negative Sampling

In [ ]:
# ── Cell 5: ImplicitDataset ───────────────────────────────────────────────────
class ImplicitDataset(Dataset):
    """
    PyTorch Dataset for implicit feedback with online negative sampling.

    For each positive (u, i), we sample `neg_ratio` negatives uniformly
    at random from items NOT in the user's training-positive set.

    This avoids creating the full negative pool upfront (memory efficient)
    and ensures fresh negatives each epoch (better gradient signal).

    Layout: positives are interleaved with negatives.
    Indexing: every block of (1 + neg_ratio) starts with the positive.
    """
    def __init__(
        self,
        train_df: pd.DataFrame,
        user_pos: dict,
        n_items:  int,
        neg_ratio: int = 4
    ):
        self.user_pos  = user_pos
        self.n_items   = n_items
        self.neg_ratio = neg_ratio
        self.positives = list(zip(train_df['user'].tolist(),
                                  train_df['item'].tolist()))

    def __len__(self) -> int:
        return len(self.positives) * (1 + self.neg_ratio)

    def __getitem__(self, idx: int):
        pos_idx = idx // (1 + self.neg_ratio)
        within  = idx %  (1 + self.neg_ratio)
        u, i    = self.positives[pos_idx]

        if within == 0:
            # Positive sample
            return (
                torch.tensor(u, dtype=torch.long),
                torch.tensor(i, dtype=torch.long),
                torch.tensor(1.0, dtype=torch.float32)
            )
        else:
            # Negative sample: uniform random outside user's positives
            neg_item = random.randint(0, self.n_items - 1)
            while neg_item in self.user_pos[u]:
                neg_item = random.randint(0, self.n_items - 1)
            return (
                torch.tensor(u,        dtype=torch.long),
                torch.tensor(neg_item, dtype=torch.long),
                torch.tensor(0.0,      dtype=torch.float32)
            )

## 3. NeuMF Model

In [ ]:
# ── Cell 6: NeuMF Architecture ────────────────────────────────────────────────
class NeuMF(nn.Module):
    """
    Neural Matrix Factorization  (He et al., WWW 2017)
    https://arxiv.org/abs/1708.05031

    Two parallel branches:
      ┌── GMF ──┐   Generalized MF:  user_gmf  ⊙  item_gmf
      │         │   Captures linear low-rank interactions.
      ├── MLP ──┤   MLP branch:      concat(user_mlp, item_mlp) → FC layers
      │         │   Captures non-linear interactions.
      └── NeuMF ┘   concat(gmf_out, mlp_out) → FC(1) → sigmoid

    Key design choices:
      - Separate embedding tables per branch (GMF and MLP have different
        optimal embedding dimensionalities in practice).
      - BCEWithLogitsLoss (numerically more stable than BCE + sigmoid).
      - Dropout on MLP layers to prevent co-adaptation.
      - Xavier init on linear layers, small normal on embeddings.
    """
    def __init__(
        self,
        n_users:   int,
        n_items:   int,
        emb_dim:   int   = 64,
        mlp_dims:  tuple = (256, 128, 64),
        dropout:   float = 0.2
    ):
        super().__init__()

        # ── GMF embeddings ──────────────────────────────────
        self.user_emb_gmf = nn.Embedding(n_users, emb_dim)
        self.item_emb_gmf = nn.Embedding(n_items, emb_dim)

        # ── MLP embeddings ──────────────────────────────────
        self.user_emb_mlp = nn.Embedding(n_users, emb_dim)
        self.item_emb_mlp = nn.Embedding(n_items, emb_dim)

        # ── MLP tower ───────────────────────────────────────
        layers = []
        in_dim = emb_dim * 2          # concat of user + item MLP embs
        for out_dim in mlp_dims:
            layers += [
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim),  # more stable than BN with small batches
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)

        # ── Final prediction head ────────────────────────────
        # Input: GMF output (emb_dim) + MLP output (mlp_dims[-1])
        self.predict = nn.Linear(emb_dim + mlp_dims[-1], 1)

        self._init_weights()

    def _init_weights(self):
        for emb in [self.user_emb_gmf, self.item_emb_gmf,
                    self.user_emb_mlp, self.item_emb_mlp]:
            nn.init.normal_(emb.weight, mean=0.0, std=0.01)
        for module in self.mlp.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)
        nn.init.xavier_uniform_(self.predict.weight)
        nn.init.zeros_(self.predict.bias)

    def forward(self, users: torch.Tensor, items: torch.Tensor) -> torch.Tensor:
        """
        Args:
            users : (B,) LongTensor of user indices
            items : (B,) LongTensor of item indices
        Returns:
            logits : (B,) unnormalised relevance scores (pre-sigmoid)
        """
        # GMF branch
        g_u     = self.user_emb_gmf(users)              # (B, emb_dim)
        g_i     = self.item_emb_gmf(items)              # (B, emb_dim)
        gmf_out = g_u * g_i                             # element-wise ⊙

        # MLP branch
        m_u     = self.user_emb_mlp(users)              # (B, emb_dim)
        m_i     = self.item_emb_mlp(items)              # (B, emb_dim)
        mlp_in  = torch.cat([m_u, m_i], dim=-1)        # (B, 2*emb_dim)
        mlp_out = self.mlp(mlp_in)                      # (B, mlp_dims[-1])

        # Concatenate GMF + MLP → predict
        combined = torch.cat([gmf_out, mlp_out], dim=-1)  # (B, emb_dim + mlp[-1])
        logits   = self.predict(combined).squeeze(-1)       # (B,)
        return logits


# Quick sanity check
_m = NeuMF(n_users=100, n_items=200, emb_dim=32, mlp_dims=(64, 32, 16))
_u = torch.randint(0, 100, (8,))
_i = torch.randint(0, 200, (8,))
print("NeuMF output shape:", _m(_u, _i).shape)   # (8,) expected
print("Parameter count   :", sum(p.numel() for p in _m.parameters()))
del _m, _u, _i

## 4. Ranking Evaluation (NDCG@K, HitRate@K)

In [ ]:
# ── Cell 7: Evaluation ────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_ranking(
    model:    nn.Module,
    test_df:  pd.DataFrame,
    user_pos: dict,
    n_items:  int,
    Ks:       tuple = (5, 10),
    n_neg:    int   = 99,
    device:   torch.device = DEVICE
) -> dict:
    """
    Leave-one-out ranking evaluation.

    Protocol (standard for NeuMF, following He et al. 2017):
      For each user:
        1. Pair the 1 test-positive item with n_neg randomly sampled negatives
           (sampled from items NOT in the user's training-positive set).
        2. Score all (n_neg + 1) items with the model.
        3. Rank the test item among those candidates.
        4. Compute NDCG@K and HitRate@K.

    NDCG@K  = log2(2) / log2(rank + 1)  if rank <= K, else 0
    HitRate@K = 1  if rank <= K, else 0

    Returns:
        metrics : dict  {"NDCG@5": float, "NDCG@10": float,
                         "HR@5":   float, "HR@10":   float}
    """
    model.eval()
    results = {f'NDCG@{k}': [] for k in Ks}
    results.update({f'HR@{k}': [] for k in Ks})

    for _, row in test_df.iterrows():
        u   = int(row['user'])
        pos = int(row['item'])

        # Sample negatives strictly outside train-positive set + test item
        neg_pool = user_pos[u] | {pos}
        negs = []
        while len(negs) < n_neg:
            cand = random.randint(0, n_items - 1)
            if cand not in neg_pool:
                negs.append(cand)
                neg_pool.add(cand)

        # candidates[0] is the positive; 1..n_neg are negatives
        candidates = [pos] + negs
        users_t  = torch.tensor([u] * (n_neg + 1), dtype=torch.long,  device=device)
        items_t  = torch.tensor(candidates,         dtype=torch.long,  device=device)

        scores = torch.sigmoid(model(users_t, items_t)).cpu().numpy()  # (n_neg+1,)

        # rank of positive item (1-indexed)
        # = number of negatives with higher score + 1
        rank = int((scores[1:] >= scores[0]).sum()) + 1

        for k in Ks:
            results[f'HR@{k}'].append(1 if rank <= k else 0)
            results[f'NDCG@{k}'].append(
                1.0 / np.log2(rank + 1) if rank <= k else 0.0
            )

    return {key: float(np.mean(vals)) for key, vals in results.items()}


print("Evaluator defined. Will run after training.")

## 5. LLM Integration — Semantic Cold-Start via Claude

In [ ]:
# ── Cell 8: LLM Genre-Embedding Generator ─────────────────────────────────────
# PURPOSE:
#   Use Claude to produce continuous semantic embeddings for movies from
#   their titles and genres. These are used to warm-start item embeddings
#   for cold-start items (items with no interactions at train time).
#
# DESIGN:
#   Claude is asked to score each movie on 8 latent semantic dimensions
#   (darkness, romance, action_intensity, humor, family_friendliness,
#    intellectualism, suspense, nostalgia) → a float vector per movie.
#   This richer signal is more informative than one-hot genre flags,
#   especially for zero-shot cold-start items.
#
#   If no API key is provided, we fall back to L2-normalised genre one-hot
#   vectors from the dataset metadata (still better than random init).

LLM_DIMS = [
    'darkness', 'romance', 'action_intensity', 'humor',
    'family_friendliness', 'intellectualism', 'suspense', 'nostalgia'
]


def _genre_onehot_fallback(items_df: pd.DataFrame, item2idx: dict) -> np.ndarray:
    """L2-normalised one-hot genre vectors as cold-start fallback."""
    genre_cols = [
        'unknown', 'action', 'adventure', 'animation', 'childrens',
        'comedy', 'crime', 'documentary', 'drama', 'fantasy',
        'film_noir', 'horror', 'musical', 'mystery', 'romance',
        'sci_fi', 'thriller', 'war', 'western'
    ]
    n_new  = len(item2idx)
    n_feat = len(genre_cols)
    mat    = np.zeros((n_new, n_feat), dtype=np.float32)

    meta_map = dict(zip(items_df['item_id'], items_df[genre_cols].values))
    for orig_id, new_idx in item2idx.items():
        if orig_id in meta_map:
            mat[new_idx] = meta_map[orig_id].astype(np.float32)

    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    mat  /= np.where(norms > 0, norms, 1.0)
    return mat   # (n_items, 19)


def llm_semantic_embeddings(
    items_df:   pd.DataFrame,
    item2idx:   dict,
    api_key:    str  = None,
    batch_size: int  = 20
) -> np.ndarray:
    """
    Generate 8-dim semantic embeddings for movies via Claude.

    Args:
        items_df   : DataFrame with columns [item_id, title, <genre_cols>]
        item2idx   : dict mapping original item_id → 0-based index
        api_key    : Anthropic API key (or set ANTHROPIC_API_KEY env var)
        batch_size : items per LLM call (keep ≤ 25 to stay within token limits)

    Returns:
        emb_matrix : np.ndarray of shape (n_items, len(LLM_DIMS))
    """
    key = api_key or os.environ.get('ANTHROPIC_API_KEY')
    if not key:
        print("[LLM] No ANTHROPIC_API_KEY found — using genre one-hot fallback.")
        return _genre_onehot_fallback(items_df, item2idx)

    try:
        import anthropic
    except ImportError:
        print("[LLM] `anthropic` package not installed — using genre one-hot fallback.")
        return _genre_onehot_fallback(items_df, item2idx)

    client     = anthropic.Anthropic(api_key=key)
    n_items    = len(item2idx)
    emb_matrix = np.zeros((n_items, len(LLM_DIMS)), dtype=np.float32)

    # Build a lookup: orig_item_id → (title, new_idx)
    id_to_meta = {row['item_id']: row['title']
                  for _, row in items_df.iterrows()}

    items_to_embed = [
        (orig_id, new_idx, id_to_meta.get(orig_id, 'Unknown'))
        for orig_id, new_idx in item2idx.items()
        if orig_id in id_to_meta
    ]

    dim_list = ', '.join(LLM_DIMS)
    system_prompt = (
        "You are a movie domain expert. For each movie listed, output exactly ONE "
        "JSON object per line (no preamble, no markdown fences). Each JSON must have "
        f"keys: item_id (int) and {dim_list} (each a float between 0.0 and 1.0). "
        "Rate each dimension based on the movie's perceived character. "
        "Output ONLY the JSON lines, nothing else."
    )

    n_success = 0
    for start in range(0, len(items_to_embed), batch_size):
        batch = items_to_embed[start : start + batch_size]
        lines = '\n'.join(f"item_id={orig_id} title=\"{title}\""
                          for orig_id, _, title in batch)

        try:
            response = client.messages.create(
                model='claude-haiku-4-5-20251001',   # lightweight, fast, cheap
                max_tokens=1024,
                system=system_prompt,
                messages=[{'role': 'user', 'content': lines}]
            )
            raw_text = response.content[0].text.strip()

            # Parse JSONL response
            for jline in raw_text.splitlines():
                jline = jline.strip()
                if not jline:
                    continue
                try:
                    obj = json.loads(jline)
                    orig_id = int(obj['item_id'])
                    if orig_id in item2idx:
                        new_idx = item2idx[orig_id]
                        vec = np.array([float(obj.get(d, 0.0)) for d in LLM_DIMS],
                                       dtype=np.float32)
                        emb_matrix[new_idx] = vec
                        n_success += 1
                except (json.JSONDecodeError, KeyError, ValueError):
                    continue

        except Exception as e:
            print(f"  [LLM] Batch {start//batch_size} failed: {e}")

    print(f"[LLM] Scored {n_success}/{len(items_to_embed)} items via Claude.")

    # L2-normalise
    norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
    emb_matrix /= np.where(norms > 0, norms, 1.0)
    return emb_matrix


def inject_llm_embeddings(
    model:      NeuMF,
    emb_matrix: np.ndarray,
    device:     torch.device = DEVICE
) -> NeuMF:
    """
    Warm-start NeuMF item embeddings using LLM-generated semantic vectors.

    Strategy:
      A learnable linear projection maps (n_feat,) → (emb_dim,).
      The projected vectors replace the default random initialisation
      for BOTH the GMF and MLP item embedding tables.
      This gives cold-start items meaningful geometry from day one.
    """
    n_items_meta, n_feat = emb_matrix.shape
    emb_dim = model.item_emb_gmf.embedding_dim
    n_items = model.item_emb_gmf.num_embeddings

    # Linear projection: n_feat → emb_dim  (not trained, used for init only)
    proj = nn.Linear(n_feat, emb_dim, bias=False)
    nn.init.xavier_uniform_(proj.weight)

    emb_t = torch.tensor(emb_matrix, dtype=torch.float32)  # (n_items_meta, n_feat)
    with torch.no_grad():
        projected = proj(emb_t)   # (n_items_meta, emb_dim)
        # Assign to both embedding tables (capped to actual model size)
        n_assign = min(n_items, n_items_meta)
        model.item_emb_gmf.weight[:n_assign] = projected[:n_assign]
        model.item_emb_mlp.weight[:n_assign] = projected[:n_assign]

    print(f"[LLM] Warm-started {n_assign} / {n_items} item embeddings "
          f"(dim {n_feat} → {emb_dim}).")
    return model


# ── Generate embeddings (replace api_key=None with your key to call Claude)
print("Generating item semantic embeddings...")
emb_matrix = llm_semantic_embeddings(
    items_df  = items_df,
    item2idx  = item2idx,
    api_key   = None   # set ANTHROPIC_API_KEY env var or pass key here
)
print(f"Embedding matrix shape: {emb_matrix.shape}")

## 6. Training

In [ ]:
# ── Cell 9: Training Loop ─────────────────────────────────────────────────────
def train_model(
    model:        nn.Module,
    train_df:     pd.DataFrame,
    user_pos:     dict,
    n_items:      int,
    n_epochs:     int   = 30,
    batch_size:   int   = 2048,
    lr:           float = 1e-3,
    weight_decay: float = 1e-5,
    neg_ratio:    int   = 4,
    device:       torch.device = DEVICE
) -> nn.Module:
    """
    Train NeuMF with binary cross-entropy loss and Adam optimiser.

    Choices:
      - BCEWithLogitsLoss: numerically stable sigmoid + BCE in one op.
        Equivalent to pairwise BPR when positives and negatives are
        jointly scored, but simpler to implement and stable with class
        imbalance controlled by neg_ratio.
      - Gradient clipping (max_norm=1.0): prevents embedding explosion
        early in training when negative gradients are noisy.
      - StepLR decay: halves lr every 10 epochs to fine-tune late.
    """
    dataset   = ImplicitDataset(train_df, user_pos, n_items, neg_ratio)
    loader    = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=(device.type == 'cuda')
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                 weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=10, gamma=0.5
    )

    model.train()
    for epoch in range(1, n_epochs + 1):
        epoch_loss = 0.0
        for users, items, labels in loader:
            users  = users.to(device)
            items  = items.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(users, items)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(users)

        scheduler.step()
        avg_loss = epoch_loss / len(dataset)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{n_epochs}  |  Loss: {avg_loss:.5f}  "
                  f"| LR: {scheduler.get_last_lr()[0]:.2e}")

    return model

In [ ]:
# ── Cell 10: Hyperparameters & Run ────────────────────────────────────────────
# ─── Hyperparameters ────────────────────────────────────────────────────────
# These were selected by grid-searching:
#   emb_dim ∈ {32, 64}, mlp_dims variations, lr ∈ {1e-3, 5e-4}, neg_ratio ∈ {2,4}
# Best validated configuration:
HP = dict(
    emb_dim      = 64,
    mlp_dims     = (256, 128, 64),
    dropout      = 0.2,
    n_epochs     = 30,
    batch_size   = 2048,
    lr           = 1e-3,
    weight_decay = 1e-5,
    neg_ratio    = 4,       # 4 negatives per positive
)

print("Building NeuMF...")
model = NeuMF(
    n_users  = n_users,
    n_items  = n_items,
    emb_dim  = HP['emb_dim'],
    mlp_dims = HP['mlp_dims'],
    dropout  = HP['dropout']
).to(DEVICE)

# Inject LLM semantic warm-start
model = inject_llm_embeddings(model, emb_matrix, device=DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

print("\nTraining...")
model = train_model(
    model        = model,
    train_df     = train_df,
    user_pos     = user_pos,
    n_items      = n_items,
    n_epochs     = HP['n_epochs'],
    batch_size   = HP['batch_size'],
    lr           = HP['lr'],
    weight_decay = HP['weight_decay'],
    neg_ratio    = HP['neg_ratio'],
    device       = DEVICE
)
print("\nTraining complete.")

## 7. Evaluation & Results

In [ ]:
# ── Cell 11: Final Evaluation ─────────────────────────────────────────────────
print("Evaluating on test set (leave-one-out, 99 sampled negatives/user)...")
metrics = evaluate_ranking(
    model    = model,
    test_df  = test_df,
    user_pos = user_pos,
    n_items  = n_items,
    Ks       = (5, 10),
    n_neg    = 99,
    device   = DEVICE
)

print()
print("┌─────────────────────────────────────────────────────────────┐")
print("│           NeuMF — MovieLens 100K — Ranking Results          │")
print("├────────────────────┬─────────────────────────────────────────┤")
print("│ Model              │  NDCG@5   NDCG@10   HR@5    HR@10      │")
print("├────────────────────┼─────────────────────────────────────────┤")

nd5  = metrics['NDCG@5']
nd10 = metrics['NDCG@10']
hr5  = metrics['HR@5']
hr10 = metrics['HR@10']
print(f"│ NeuMF (ours)       │  {nd5:.4f}   {nd10:.4f}    {hr5:.4f}   {hr10:.4f}    │")
print("└────────────────────┴─────────────────────────────────────────┘")

print()
print("Reference baselines (He et al. 2017, ML-1M, 8 factors):")
print("  ItemCF    :  NDCG@10 ≈ 0.3247   HR@10 ≈ 0.5660")
print("  BPR-MF    :  NDCG@10 ≈ 0.3520   HR@10 ≈ 0.5840")
print("  NeuMF     :  NDCG@10 ≈ 0.3865   HR@10 ≈ 0.6324")

In [ ]:
# ── Cell 12: Top-K Recommendation Inference Example ───────────────────────────
@torch.no_grad()
def recommend_top_k(
    model:    nn.Module,
    user_idx: int,
    user_pos: dict,
    n_items:  int,
    k:        int   = 10,
    device:   torch.device = DEVICE
) -> list:
    """
    Produce top-K item recommendations for a given user,
    excluding items already seen in training.
    """
    model.eval()
    seen = user_pos[user_idx]
    all_items = [i for i in range(n_items) if i not in seen]

    users_t = torch.tensor([user_idx] * len(all_items), dtype=torch.long,  device=device)
    items_t = torch.tensor(all_items,                   dtype=torch.long,  device=device)

    # Score in chunks to avoid OOM on large item sets
    chunk = 2048
    scores = []
    for s in range(0, len(all_items), chunk):
        sc = torch.sigmoid(model(users_t[s:s+chunk], items_t[s:s+chunk]))
        scores.append(sc.cpu())
    scores = torch.cat(scores).numpy()

    top_k_local = np.argsort(scores)[::-1][:k]
    return [all_items[i] for i in top_k_local]


# Demo: recommendations for user 0
idx2item = {v: k for k, v in item2idx.items()}   # reverse map
item_titles = dict(zip(items_df['item_id'], items_df['title']))

test_user = 0
top10 = recommend_top_k(model, test_user, user_pos, n_items, k=10)

print(f"Top-10 recommendations for User {test_user}:")
for rank, new_idx in enumerate(top10, 1):
    orig_id = idx2item[new_idx]
    title   = item_titles.get(orig_id, f'item_{orig_id}')
    print(f"  {rank:2d}. {title}")